Bibliotecas necessárias.

In [22]:
# Importar bibliotecas necessárias
import pandas as pd
import re
import gc
from google.colab import userdata

Funções auxiliares.

In [23]:
# Função para remover a parte numérica inicial e os possíveis sublinhado que a segue.
def clean_category(category):
  # Dicionário de correções de termos jurídicos.
  corrections_dict = {
      'tributario': 'tributário'
  }
  cleaned_numeric_part = re.sub(r'^\d+_?', '', category)

  # Substituir todos os sublinhados por espaços
  final_string = cleaned_numeric_part.replace('_', ' ')

  # Aplicar correções.
  words = final_string.split()
  corrected_words = []
  for word in words:
    # Verifica se a palavra está no dicionário de correções e aplica
    corrected_words.append(corrections_dict.get(word, word))
  final_string = ' '.join(corrected_words)

  # Converter para modo título: primeira letra de cada palavra maiúscula, as demais minúsculas,
  # com as exceções de alguns artigos e preosições.
  lowercase_exceptions = {'e', 'o', 'a', 'de', 'do', 'dos', 'da', 'das'}
  processed_words = []
  for word in final_string.split():
    if word.lower() in lowercase_exceptions:
      processed_words.append(word.lower())
    else:
      processed_words.append(word.title())
  # Retorno.
  return ' '.join(processed_words)

# Função que recebe a url e retorna um Dataframe.
def load_dataframe(url):
   return pd.read_json(url, lines=True)

# Função recuperar todas as categorias de um Dataframe de forma distinta.
def get_unique_categories(df):
  return df['category'].unique()

# Função para juntar dois Dataframes por um campo chave e opcionalmente selecionar colunas.
def merge_dataframes(df1, df2, key, columns=None):
  merged_df = pd.merge(df1, df2, on=key, how='inner')
  if columns is not None:
    return merged_df[columns]
  return merged_df

# Função para converter um dataframe em jsonl e salvá-lo com arquivo no colab.
def save_df_to_jsonl(df: pd.DataFrame, filename: str):
  try:
    # Converte o DataFrame para JSONL
    # orient='records' serializa cada linha como um objeto JSON
    # lines=True garante que cada objeto JSON esteja em uma nova linha (formato JSONL)
    # json_format='plain' evita o escape de caracteres não-ASCII, preservando a acentuação.
    #jsonl_output = df.to_json(orient='records', lines=True, json_format='plain')

    # Exportando o dataframe para JSONL de forma correta
    df.to_json(
        filename,
        orient='records',
        lines=True,
        force_ascii=False  # <--- Não converter para ascii.
    )
    print(f"DataFrame salvo com sucesso em {filename} no formato JSONL.")
  except Exception as e:
    print(f"Ocorreu um erro ao salvar o DataFrame em JSONL: {e}")

Funções que povoam os Dataframes com as questões e respostas disponíveis no github.

In [24]:
# Dicionário de edições.
edicoes_oab = {
    "39": "2023.1",
    "40": "2023.2"
}

# Dataframe questions.
def load_questions():
  # URL das questões.
  url_questions = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/main/data/oab_bench/question.jsonl"

  # Carregar um Dataframe com as perguntas a partir do arquivo .jsonl.
  df_questions = load_dataframe(url_questions)

  # Inserir uma coluna para enumerar as questões, com contagem a partir do número 1, uma vez que a contagem de linhas do python é a partir do 0.
  df_questions.insert(0, 'question', range(1, len(df_questions)+1))

  # Realizar limpeza no campo das categorias.
  df_questions['category'] = df_questions['category'].apply(clean_category)

  # retorno da Dataframe.
  return df_questions

# Dataframe guidelines.
def load_guidelines():
  # URL das respostas de referência.
  url_guidelines = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/refs/heads/main/data/oab_bench/reference_answer/guidelines.jsonl"
  # Carregar um Dataframe com as respostas a partir do arquivo .jsonl.
  return load_dataframe(url_guidelines)

# Dataframe da junção dos DataFrames df_questions e df_guidelines em um único DataFrame usando o question_id de ambos como atributo da interseção.
# Neste Dataframe dá para ver a pergunta e a resposta da linha guia juntas.
# E, também personalizada as colunas que serão apresentadas.
def load_questions_and_guidelines():
  # Carregar os Dataframes.
  df_questions = load_questions()
  df_guidelines = load_guidelines()

  # Chave do inner join
  key = 'question_id'
  # Colunas selecionar para projeção no Dataframe.
  columns=['question', 'question_id', 'category', 'statement', 'turns', 'system', 'choices']

  # Junção e retorno do Dataframe
  return merge_dataframes(df_questions, df_guidelines, key, columns)

# Dataframe com um subconjunto das perguntas e linhas guias.
# O índice do Dataframe começa do 0 (zero), portanto, a linha 1 é a 0, a 2 é a 1, etc.
# iloc seleciona um intervalo fechado à esquerda e aberto à direita
def load_my_questions_and_guidelines(question_min, question_max):
  # Remover 1 und dando o descontando, uma vez que o valor passado é o número da questão para o ser humano
  # e não a contagem do python.
  question_min -= 1
  df_my_questions = load_questions_and_guidelines().iloc[question_min:question_max].copy()

  # Realizar limpeza das informações da coluna turns, pois é um array.
  # E, só preciso da infomação, quando houver, da primeira posição.
  df_my_questions['turns'] = df_my_questions['turns'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else '')

  # Limpeza também a coluna choices, pois é um dicionário que só preciso do campo turns.
  df_my_questions['choices'] = df_my_questions['choices'].apply(lambda x: x[0]['turns'][0] if isinstance(x, list) and len(x) > 0 and 'turns' in x[0] and isinstance(x[0]['turns'], list) and len(x[0]['turns']) > 0 else '')

  # Adição da edição das provas em um novo campo, utilizando como referência o dicionário de edições de provas da OAB.
  df_my_questions['edicao'] = df_my_questions['question_id'].apply(lambda x: x[:2])

  # Adição da edição das provas em um novo campo, utilizando como referência o dicionário de edições de provas da OAB.
  df_my_questions['ano_semestre'] = df_my_questions['question_id'].apply(lambda x: edicoes_oab.get(x[:2], 'Edição Desconhecida'))

  # Retornar dataframe ajustado.
  return df_my_questions

# Excluir Dataframes desnecessários.
def delete_obsoletes_objects():
  # Excluir Dataframes que não são mais necessários, apenas se existirem no escopo global.
  if 'df_questions' in globals():
    del df_questions
  if 'df_guidelines' in globals():
    del df_guidelines
  gc.collect()

Respostas de referências às questões anteriores e disponíveis no mesmo repositório do github.

In [ ]:
# Meu sub-grupo de questões e respostas.
# Subconjunto das questões de 31 a 40.
question_min = 31
question_max = 40
df_my_questions = load_my_questions_and_guidelines(question_min, question_max)

# Salvar arquivo jsonl.
save_df_to_jsonl(df_my_questions, 'my_questions.jsonl')

DataFrame salvo com sucesso em my_questions.jsonl no formato JSONL.


Função para definir o nível de complexidade das questões.

In [ ]:
# Classificar os níveis das perguntas.
# Após analise manual e também com ajuda de algumas IAs percebi que todas as questões com texto
# no turns do subconjunto que peguei ficou no nível 3.
# E, atribui o nível 1 a quem não tem texto algum em turns.
# Os níveis são:
#   1: Estagiário;
#   2: Analista;
#   3: Juiz;
#   4: Ministro.
def classify_difficulty_level():
  for index, row in df_my_questions.iterrows():
      # Verifica se a coluna 'turns' está vazia ou contém apenas espaços em branco
      if not row['turns'] or str(row['turns']).strip() == '':
          df_my_questions.loc[index, 'level'] = 1
      else:
          df_my_questions.loc[index, 'level'] = 3
  # Converter a coluna 'level' para o tipo inteiro após o preenchimento
  df_my_questions['level'] = df_my_questions['level'].astype(int)

Chamar a função que adiciona o nível de dificuldade no dataframe df_my_questions.

In [ ]:
classify_difficulty_level()

Liberar espeço não necessário em memória.

In [ ]:
delete_obsoletes_objects()

In [ ]:
df_my_questions

,question,question_id,category,statement,turns,system,choices,edicao,ano_semestre,level
30,31,39_direito_tributario_peca_profissional,Direito Tributário,"PEÇA PRÁTICO-PROFISSIONAL\n\nPedro de Camões, ...",,Você é um bacharel em direito que está realiza...,"A medida judicial cabível é a ação anulatória,...",39,2023.1,1
31,32,39_direito_tributario_questao_1,Direito Tributário,QUESTÃO\n\nA sociedade empresária Metalúrgica ...,"A partir de quando se deve contar, no caso con...",Você é um bacharel em direito que está realiza...,O prazo para oferta dos embargos à execução fi...,39,2023.1,3
32,33,39_direito_tributario_questao_2,Direito Tributário,QUESTÃO\n\nA sociedade empresária Tudo Verde L...,A qual dos municípios o ISS de jardinagem é de...,Você é um bacharel em direito que está realiza...,O ISS de jardinagem é devido ao Município Beta...,39,2023.1,3
33,34,39_direito_tributario_questao_3,Direito Tributário,QUESTÃO\n\nA Administração Tributária Federal ...,É válida a exigência de depósito do montante c...,Você é um bacharel em direito que está realiza...,"Não, pois é inconstitucional a exigência de de...",39,2023.1,3
34,35,39_direito_tributario_questao_4,Direito Tributário,QUESTÃO\n\nA sociedade empresária Faz Tudo Ltd...,Está correto o argumento da necessidade de not...,Você é um bacharel em direito que está realiza...,"Não está correto, porque a entrega de declaraç...",39,2023.1,3
35,36,40_direito_administrativo_peca_profissional,Direito Administrativo,PEÇA PRÁTICO-PROFISSIONAL\n\nO Ministério Públ...,,Você é um bacharel em direito que está realiza...,O(a) examinando(a) deve apresentar recurso de ...,40,2023.2,1
36,37,40_direito_administrativo_questao_1,Direito Administrativo,QUESTÃO\n\nA sociedade empresária Sagaz S.A. e...,Há necessidade de demonstração do elemento sub...,Você é um bacharel em direito que está realiza...,Não. A responsabilização administrativa por at...,40,2023.2,3
37,38,40_direito_administrativo_questao_2,Direito Administrativo,QUESTÃO\n\nDeterminada informação de interesse...,É lícita a cobrança efetuada pelo órgão respon...,Você é um bacharel em direito que está realiza...,Não. A submissão e o processamento de pedido d...,40,2023.2,3
38,39,40_direito_administrativo_questao_3,Direito Administrativo,QUESTÃO\n\nCerta Secretaria do Estado Alfa fez...,É possível a utilização do sistema de registro...,Você é um bacharel em direito que está realiza...,Sim. A Administração poderá contratar a execuç...,40,2023.2,3
39,40,40_direito_administrativo_questao_4,Direito Administrativo,"QUESTÃO\n\nRecentemente, Iná foi aprovada em c...",A aprovação de Iná no mencionado concurso impo...,Você é um bacharel em direito que está realiza...,"Não. Iná foi aprovada para emprego público, ao...",40,2023.2,3


In [ ]:
load_questions_and_guidelines()

,question,question_id,category,statement,turns,system,choices
0,1,39_direito_administrativo_peca_profissional,Direito Administrativo,PEÇA PRÁTICO-PROFISSIONAL\n\nEm fevereiro de 2...,[],Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['A peça a ser apresent..."
1,2,39_direito_administrativo_questao_1,Direito Administrativo,QUESTÃO\n\nA União fez publicar um edital de l...,[O edital em questão deveria contemplar a matr...,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Sim. Nos contratos de..."
2,3,39_direito_administrativo_questao_2,Direito Administrativo,QUESTÃO\n\nA sociedade empresária Alfa foi con...,[A contratada tem direito à extinção do contra...,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Sim, a sociedade empr..."
3,4,39_direito_administrativo_questao_3,Direito Administrativo,QUESTÃO\n\nJaqueline é servidora pública ocupa...,"[Jaqueline, como agente público responsável pe...",Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Sim, Jaqueline, como ..."
4,5,39_direito_administrativo_questao_4,Direito Administrativo,QUESTÃO\n\nApós realizar pedido administrativo...,[Qual o prazo para a interposição do recurso a...,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Considerando que não ..."
...,...,...,...,...,...,...,...
205,206,44_direito_penal_peca_profissional,Direito Penal,"No dia 10/1/2024, Aluísio, entregador, foi rea...",[],Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['De acordo com as info..."
206,207,44_direito_penal_questao_1,Direito Penal,"Carlos, com a intenção de obter vantagem indev...",[A) Qual a tese de Direito Penal deve ser sust...,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['A questão exige do ex..."
207,208,44_direito_penal_questao_2,Direito Penal,"Manoela, um dia antes de completar 18 anos, en...","[A) A fim de afastar, completamente, a respons...",Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['A questão exige do ex..."
208,209,44_direito_penal_questao_3,Direito Penal,"Karina e Daniel, casados, celebraram um contra...",[A) Qual a tese de Direito Processual cabível ...,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['A questão exige do ex..."


In [ ]:
df_my_questions.head()

,question,question_id,category,statement,turns,system,choices,edicao,ano_semestre,level
30,31,39_direito_tributario_peca_profissional,Direito Tributário,"PEÇA PRÁTICO-PROFISSIONAL\n\nPedro de Camões, ...",,Você é um bacharel em direito que está realiza...,"A medida judicial cabível é a ação anulatória,...",39,2023.1,1
31,32,39_direito_tributario_questao_1,Direito Tributário,QUESTÃO\n\nA sociedade empresária Metalúrgica ...,"A partir de quando se deve contar, no caso con...",Você é um bacharel em direito que está realiza...,O prazo para oferta dos embargos à execução fi...,39,2023.1,3
32,33,39_direito_tributario_questao_2,Direito Tributário,QUESTÃO\n\nA sociedade empresária Tudo Verde L...,A qual dos municípios o ISS de jardinagem é de...,Você é um bacharel em direito que está realiza...,O ISS de jardinagem é devido ao Município Beta...,39,2023.1,3
33,34,39_direito_tributario_questao_3,Direito Tributário,QUESTÃO\n\nA Administração Tributária Federal ...,É válida a exigência de depósito do montante c...,Você é um bacharel em direito que está realiza...,"Não, pois é inconstitucional a exigência de de...",39,2023.1,3
34,35,39_direito_tributario_questao_4,Direito Tributário,QUESTÃO\n\nA sociedade empresária Faz Tudo Ltd...,Está correto o argumento da necessidade de not...,Você é um bacharel em direito que está realiza...,"Não está correto, porque a entrega de declaraç...",39,2023.1,3


In [ ]:
df_questions = load_guidelines()
df_guidelines = load_questions()

In [ ]:
df_questions.head()

,question_id,answer_id,model_id,choices,tstamp
0,39_direito_administrativo_peca_profissional,2703a8c66fd84114b60e7ab713e4a9cc,guidelines,"[{'index': 0, 'turns': ['A peça a ser apresent...",1.686287e+09
1,39_direito_administrativo_questao_1,1ba691a0a6af496eb6b9a2b83b1ad689,guidelines,"[{'index': 0, 'turns': ['Sim. Nos contratos de...",NaN
2,39_direito_administrativo_questao_2,7524549417e94e03953b53d4e4964ad3,guidelines,"[{'index': 0, 'turns': ['Sim, a sociedade empr...",NaN
3,39_direito_administrativo_questao_3,8c128077f7794cc4b47e0cb659aedc01,guidelines,"[{'index': 0, 'turns': ['Sim, Jaqueline, como ...",NaN
4,39_direito_administrativo_questao_4,d61e1b8372534c19bf11f04614ffb3fa,guidelines,"[{'index': 0, 'turns': ['Considerando que não ...",NaN


In [ ]:
df_guidelines.head()

,question,question_id,category,statement,turns,values,system
0,1,39_direito_administrativo_peca_profissional,Direito Administrativo,PEÇA PRÁTICO-PROFISSIONAL\n\nEm fevereiro de 2...,[],[5.0],Você é um bacharel em direito que está realiza...
1,2,39_direito_administrativo_questao_1,Direito Administrativo,QUESTÃO\n\nA União fez publicar um edital de l...,[O edital em questão deveria contemplar a matr...,"[0.65, 0.6000000000000001]",Você é um bacharel em direito que está realiza...
2,3,39_direito_administrativo_questao_2,Direito Administrativo,QUESTÃO\n\nA sociedade empresária Alfa foi con...,[A contratada tem direito à extinção do contra...,"[0.65, 0.6000000000000001]",Você é um bacharel em direito que está realiza...
3,4,39_direito_administrativo_questao_3,Direito Administrativo,QUESTÃO\n\nJaqueline é servidora pública ocupa...,"[Jaqueline, como agente público responsável pe...","[0.65, 0.6000000000000001]",Você é um bacharel em direito que está realiza...
4,5,39_direito_administrativo_questao_4,Direito Administrativo,QUESTÃO\n\nApós realizar pedido administrativo...,[Qual o prazo para a interposição do recurso a...,"[0.6000000000000001, 0.65]",Você é um bacharel em direito que está realiza...
